# Bài 4.1 · Dùng model pretrained trên ảnh thật

**Câu hỏi của buổi này:** một model đã học trên ImageNet nhìn một bức ảnh chụp thật ra sao, và vì sao chỉ sai một bước tiền xử lý nhỏ cũng đủ làm nó đoán sai?

Dữ liệu là **Imagenette**: 10 lớp lấy từ ImageNet, ảnh chụp thật. Vì 10 lớp này nằm sẵn trong 1000 lớp mà ResNet-50 đã học, ta dùng model ngay, **không train gì cả**.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng 10 lớp, mỗi lớp một ảnh |
| 2. Model và hợp đồng | bảng các bước tiền xử lý, ảnh trước và sau khi xử lý |
| 3. Top-5 | ảnh kèm biểu đồ 5 lớp model tin nhất |
| 4. Accuracy | bảng accuracy, biểu đồ theo lớp, những ảnh model chọn lớp khác |
| 5. Làm sai hợp đồng | bảng và biểu đồ: mỗi lỗi làm accuracy tụt bao nhiêu, ảnh model thật sự nhìn thấy |
| 6. Batch và float16 | biểu đồ tốc độ và bộ nhớ |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 2 phút. Bài giảng: [ai-canuc80k · bài 4.1](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/4.1).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style
    if fmt:
        st = st.format(fmt)
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, random, tarfile, urllib.request
from pathlib import Path
from collections import Counter
import torch, torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset

random.seed(0); np.random.seed(0); torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
table(pd.DataFrame([('torch', torch.__version__), ('torchvision', torchvision.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Mỗi thư mục con của Imagenette mang **mã WordNet** của lớp. Model trả về 1000 logits theo thứ tự của ImageNet, nên ta cần bảng đổi mã ấy sang **chỉ số trong 1000 lớp**.

In [ ]:
URL = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz'
root = Path('/tmp/imagenette')
if not (root / 'imagenette2-320').exists():
    root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(URL, root / 'data.tgz')
    with tarfile.open(root / 'data.tgz') as f:
        f.extractall(root)
data_dir = root / 'imagenette2-320'

CLASSES = {  # mã WordNet: (chỉ số ImageNet, tên tiếng Việt)
    'n01440764': (0, 'cá tench'), 'n02102040': (217, 'chó springer'), 'n02979186': (482, 'máy cassette'),
    'n03000684': (491, 'cưa máy'), 'n03028079': (497, 'nhà thờ'), 'n03394916': (566, 'kèn French horn'),
    'n03417042': (569, 'xe rác'), 'n03425413': (571, 'cây xăng'), 'n03445777': (574, 'bóng golf'),
    'n03888257': (701, 'dù lượn')}
view_ds = torchvision.datasets.ImageFolder(data_dir / 'val')
imagenet_idx = torch.tensor([CLASSES[w][0] for w in view_ds.classes])
vi = [CLASSES[w][1] for w in view_ds.classes]
counts = np.bincount(view_ds.targets)

table(pd.DataFrame({'lớp': vi, 'mã WordNet': view_ds.classes, 'chỉ số ImageNet': imagenet_idx.tolist(), 'số ảnh validation': counts}),
      'Mười lớp của Imagenette')
check('đủ 10 lớp', len(view_ds.classes) == 10)
check(f'{len(view_ds)} ảnh validation, lớp ít nhất {counts.min()}, nhiều nhất {counts.max()}', len(view_ds) > 3000)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for k, ax in enumerate(axes.flat):
    img, _ = view_ds[view_ds.targets.index(k)]
    ax.imshow(img); ax.set_title(vi[k]); ax.axis('off')
fig.suptitle('Mỗi lớp một ảnh: ảnh chụp thật, nền và góc chụp rất khác nhau', y=1.0)
plt.tight_layout(); plt.show()

## 2. Model và hợp đồng của nó

Một model pretrained đi kèm một **hợp đồng**: ảnh phải được đưa vào đúng như lúc model học. `weights.transforms()` ghi lại hợp đồng ấy.

In [ ]:
weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
with quiet():
    model = torchvision.models.resnet50(weights=weights).to(device).eval()
prep = weights.transforms()
categories = weights.meta['categories']
mean, std = prep.mean, prep.std

table(pd.DataFrame([
    ('1. thu ảnh về', f'cạnh ngắn {prep.resize_size[0]} pixel'),
    ('2. cắt giữa', f'{prep.crop_size[0]} × {prep.crop_size[0]}'),
    ('3. thứ tự kênh', 'RGB (ảnh đọc bằng PIL)'),
    ('4. đổi sang số thực', 'mỗi pixel trong [0, 1]'),
    ('5. chuẩn hóa từng kênh', f'trừ mean {[round(m, 3) for m in mean]}, chia std {[round(s, 3) for s in std]}'),
], columns=['bước', 'giá trị']), 'Hợp đồng tiền xử lý của ResNet-50 (IMAGENET1K_V2)')

n_params = sum(p.numel() for p in model.parameters())
check(f'{n_params:,} tham số, khoảng {n_params * 4 / 2**20:.0f} MB ở float32', n_params > 2e7)
check('model ở chế độ eval (BatchNorm dùng thống kê đã lưu)', not model.training)
check('1000 lớp đầu ra', len(categories) == 1000)

In [ ]:
img, y = view_ds[view_ds.targets.index(7)]
resize_crop = T.Compose([T.Resize(prep.resize_size[0]), T.CenterCrop(prep.crop_size[0])])
cropped = resize_crop(img)
x = prep(img)
raw = T.ToTensor()(cropped)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={'width_ratios': [1, 1, 1.4]})
axes[0].imshow(img); axes[0].set_title(f'ảnh gốc: rộng {img.size[0]}, cao {img.size[1]}'); axes[0].axis('off')
axes[1].imshow(cropped); axes[1].set_title('sau khi thu và cắt: 224×224'); axes[1].axis('off')
axes[2].hist(raw.flatten(), bins=60, alpha=.75, label='trước chuẩn hóa: [0, 1]', color=C['gray'])
axes[2].hist(x.flatten(), bins=60, alpha=.75, label='sau chuẩn hóa: quanh 0', color=C['blue'])
axes[2].set_title('giá trị pixel đưa vào model'); axes[2].set_xlabel('giá trị'); axes[2].set_ylabel('số giá trị'); axes[2].legend()
plt.tight_layout(); plt.show()

table(pd.DataFrame({'kênh': ['R', 'G', 'B'], 'mean sau chuẩn hóa': x.mean((1, 2)).numpy(), 'std sau chuẩn hóa': x.std((1, 2)).numpy()}),
      'Tensor đưa vào model sau chuẩn hóa: mỗi kênh dời về quanh 0', fmt={'mean sau chuẩn hóa': '{:.2f}', 'std sau chuẩn hóa': '{:.2f}'})
check(f'tensor có shape {tuple(x.shape)}', tuple(x.shape) == (3, 224, 224))

## 3. Top-5: model tin những lớp nào?

Softmax trên 1000 logits cho một phân phối xác suất. Thanh **xanh lá** là lớp đúng; thanh xám là lớp khác. Model chưa từng được bảo rằng đề chỉ có 10 lớp, nên nó có thể chọn một lớp gần nghĩa nằm ngoài danh sách.

In [ ]:
def top5(img):
    with torch.inference_mode():
        p = model(prep(img)[None].to(device)).softmax(1)[0].cpu()
    return p.topk(5)

picks = random.Random(1).sample(range(len(view_ds)), 6)
fig, axes = plt.subplots(2, 6, figsize=(18, 6.4), gridspec_kw={'height_ratios': [3, 2]})
hits, tops = [], []
for j, i in enumerate(picks):
    img, y = view_ds[i]
    top = top5(img)
    hits.append(int(top.indices[0]) == int(imagenet_idx[y]))
    tops.append(float(top.values[0]))
    axes[0, j].imshow(resize_crop(img)); axes[0, j].axis('off')
    axes[0, j].set_title(f"thật: {vi[y]}\n{'✓ đúng' if hits[-1] else '✗ sai'} ở top-1", color=C['green'] if hits[-1] else C['clay'])
    colors = [C['green'] if k == imagenet_idx[y] else C['gray'] for k in top.indices]
    axes[1, j].barh(range(5)[::-1], top.values, color=colors); bar_labels(axes[1, j])
    axes[1, j].set_yticks(range(5)[::-1], [categories[k][:16] for k in top.indices], fontsize=9)
    axes[1, j].set_xlim(0, 1); axes[1, j].grid(axis='y', visible=False)
plt.tight_layout(); plt.show()
note(f'{sum(hits)}/{len(hits)} ảnh có lớp đúng ở top-1. Xác suất của lớp đứng đầu chỉ từ {min(tops):.2f} đến {max(tops):.2f}: phần còn lại chia cho các lớp gần nghĩa.')

## 4. Accuracy trên toàn bộ tập validation

Hai cách chấm:

- **1000 lớp**: đúng khi lớp model tin nhất trong cả 1000 lớp chính là lớp của ảnh.
- **10 lớp**: chỉ so 10 logits của Imagenette, như khi biết trước đề chỉ có 10 đáp án.

In [ ]:
def evaluate(transform, train_mode=False, batch=128, n=None):
    ds = torchvision.datasets.ImageFolder(data_dir / 'val', transform=transform)
    if n is not None:
        ds = Subset(ds, np.random.default_rng(0).permutation(len(ds))[:n])
    model.train(train_mode)
    pred1000, pred10, ys = [], [], []
    with torch.inference_mode():
        for xb, yb in DataLoader(ds, batch, shuffle=False, num_workers=4, pin_memory=True):
            logits = model(xb.to(device)).cpu()
            pred1000.append(logits.argmax(1)); pred10.append(logits[:, imagenet_idx].argmax(1)); ys.append(yb)
    model.eval()
    p1000, p10, y = torch.cat(pred1000), torch.cat(pred10), torch.cat(ys)
    return (p1000 == imagenet_idx[y]).float().mean().item(), (p10 == y).float().mean().item(), p1000, p10, y

t = time.time()
acc1000, acc10, p1000, p10, y_all = evaluate(prep)
table(pd.DataFrame([('1000 lớp', acc1000), ('chỉ 10 lớp Imagenette', acc10)], columns=['cách chấm', 'accuracy']),
      f'Accuracy trên {len(y_all)} ảnh validation ({time.time() - t:.0f} giây, không train)', fmt={'accuracy': '{:.3f}'})
check('chọn trong 10 lớp đúng trên 95% ảnh', acc10 > 0.95)

In [ ]:
per_class = pd.DataFrame({'lớp': vi,
                          '10 lớp': [(p10[y_all == k] == k).float().mean().item() for k in range(10)],
                          '1000 lớp': [(p1000[y_all == k] == imagenet_idx[k]).float().mean().item() for k in range(10)]})
ax = per_class.set_index('lớp')[['10 lớp', '1000 lớp']].plot.barh(figsize=(9, 6), color=[C['blue'], C['clay']], width=.8)
ax.set_xlim(0, 1.12); ax.set_xlabel('accuracy'); ax.set_ylabel(''); ax.set_title('Accuracy theo từng lớp'); ax.invert_yaxis()
ax.legend(title='chọn trong', loc='upper left', bbox_to_anchor=(1, 1)); bar_labels(ax)
plt.tight_layout(); plt.show()
worst = per_class.sort_values('1000 lớp').iloc[0]
note(f"Lớp khó nhất khi chấm 1000 lớp là <b>{worst['lớp']}</b>: {worst['1000 lớp']:.2f}, dù chọn trong 10 lớp vẫn đạt {worst['10 lớp']:.2f}.")

In [ ]:
lost = (p1000 != imagenet_idx[y_all]) & (p10 == y_all)        # sai ở 1000 lớp nhưng đúng ở 10 lớp
pairs = Counter((vi[int(a)], categories[int(b)]) for a, b in zip(y_all[lost], p1000[lost]))
table(pd.DataFrame([(a, b, n) for (a, b), n in pairs.most_common(8)], columns=['lớp thật', 'model chọn trong 1000 lớp', 'số ảnh']),
      'Khi chấm 1000 lớp, model hay chọn lớp nào thay thế?')
note(f'{int(lost.sum())} ảnh bị tính sai ở 1000 lớp nhưng đúng khi chỉ chọn trong 10 lớp: model chọn một lớp ImageNet khác.')

idx_lost = [int(torch.nonzero(lost & (y_all == k)).flatten()[0]) for k in range(10) if (lost & (y_all == k)).any()][:6]
fig, axes = plt.subplots(1, 6, figsize=(18, 3.6))
for ax, i in zip(axes, idx_lost):
    img, y = view_ds[i]
    ax.imshow(img); ax.axis('off'); ax.set_title(f'thật: {vi[y]}\nmodel: {categories[int(p1000[i])][:18]}', fontsize=10)
plt.tight_layout(); plt.show()

## 5. Làm sai hợp đồng từng chỗ một

Mỗi dòng chỉ đổi **một** điều so với cách đúng; model, trọng số và ảnh giữ nguyên. Chấm trên cùng 1000 ảnh validation.

- **Quên chuẩn hóa**: đầu vào trong [0, 1] thay vì quanh 0.
- **Đảo kênh**: đưa BGR (thứ tự của OpenCV) vào model học bằng RGB.
- **Để `train()`**: BatchNorm dùng thống kê của chính batch 8 ảnh thay vì thống kê đã lưu.
- **Ảnh 64 pixel**: thu về 64 rồi phóng lại 224, chi tiết mất hết.

In [ ]:
base = [T.Resize(prep.resize_size[0]), T.CenterCrop(prep.crop_size[0]), T.ToTensor()]
norm = T.Normalize(mean, std)
cases = [
    ('đúng hợp đồng', T.Compose(base + [norm]), False, 128),
    ('quên chuẩn hóa', T.Compose(base), False, 128),
    ('đảo kênh RGB → BGR', T.Compose(base + [norm, T.Lambda(lambda x: x.flip(0))]), False, 128),
    ('để train(), batch 8', T.Compose(base + [norm]), True, 8),
    ('ảnh 64 pixel', T.Compose([T.Resize(64), T.CenterCrop(64), T.Resize(224)] + base[2:] + [norm]), False, 128),
]
rows = []
for name, tf, train_mode, b in cases:
    a1000, a10, *_ = evaluate(tf, train_mode, b, n=1000)
    rows.append((name, a1000, a10))
res = pd.DataFrame(rows, columns=['cách', 'acc 1000 lớp', 'acc 10 lớp'])
res['tụt so với đúng'] = res['acc 1000 lớp'] - res['acc 1000 lớp'].iloc[0]
table(res, 'Mỗi lỗi tiền xử lý làm accuracy đổi bao nhiêu (1000 ảnh)',
      fmt={'acc 1000 lớp': '{:.3f}', 'acc 10 lớp': '{:.3f}', 'tụt so với đúng': '{:+.3f}'})

ax = res.set_index('cách')['acc 1000 lớp'].plot.barh(figsize=(9, 3.8), color=[C['blue']] + [C['clay']] * 4, width=.7)
ax.set_xlim(0, 1.08); ax.set_xlabel('accuracy 1000 lớp'); ax.set_ylabel(''); ax.invert_yaxis()
ax.set_title('Đúng hợp đồng (xanh) và bốn cách làm sai'); bar_labels(ax, '{:.3f}')
plt.tight_layout(); plt.show()
bad = res.iloc[1:].sort_values('acc 1000 lớp').iloc[0]
mild = res.iloc[1:].sort_values('acc 1000 lớp').iloc[-1]
note(f"Lỗi nặng nhất: <b>{bad['cách']}</b>, accuracy còn {bad['acc 1000 lớp']:.3f} ({bad['tụt so với đúng']:+.3f}). "
     f"Lỗi nhẹ nhất: <b>{mild['cách']}</b> ({mild['tụt so với đúng']:+.3f}); hình dưới cho thấy lỗi nhẹ vẫn làm model kém tự tin.")

In [ ]:
img, y = view_ds[view_ds.targets.index(1)]
show_cases = [c for c in cases if not c[2]]                       # bốn cách đổi ảnh đầu vào
fig, axes = plt.subplots(1, 4, figsize=(16, 4.6))
conf = []
for ax, (name, tf, _, _) in zip(axes, show_cases):
    xin = tf(img)
    with torch.inference_mode():
        p = model(xin[None].to(device)).softmax(1)[0].cpu()
    k = int(p.argmax())
    conf.append((name, float(p[imagenet_idx[y]])))
    view = xin if name == 'quên chuẩn hóa' else xin * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None]
    ax.imshow(view.clamp(0, 1).permute(1, 2, 0)); ax.axis('off')
    ok = k == imagenet_idx[y]
    ax.set_title(f'{name}\n→ {categories[k][:20]} ({p[k]:.2f})', color=C['green'] if ok else C['clay'], fontsize=10)
fig.suptitle(f'Cùng một ảnh “{vi[y]}”: thứ model thật sự nhìn thấy, và lớp nó chọn', y=1.02)
plt.tight_layout(); plt.show()
note('Xác suất model dành cho lớp đúng: ' + ', '.join(f'{n} <b>{c:.2f}</b>' for n, c in conf) + '.')

## 6. Batch, tốc độ và bộ nhớ

Trên GPU, batch lớn hơn tận dụng song song tốt hơn nên số ảnh mỗi giây tăng tới khi bão hòa; bộ nhớ đỉnh tăng gần tỉ lệ với batch. `autocast` float16 tốn ít bộ nhớ hơn và nhanh hơn khi batch đủ lớn.

In [ ]:
def bench(batch, half):
    x = torch.randn(batch, 3, 224, 224, device=device)
    if device == 'cuda':
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode(), torch.autocast(device, dtype=torch.float16, enabled=half and device == 'cuda'):
        model(x)
        if device == 'cuda': torch.cuda.synchronize()
        t = time.time()
        for _ in range(5):
            model(x)
        if device == 'cuda': torch.cuda.synchronize()
    sec = (time.time() - t) / 5
    mem = torch.cuda.max_memory_allocated() / 2**20 if device == 'cuda' else np.nan
    return batch / sec, mem

rows = [(('float16' if half else 'float32'), b, *bench(b, half))
        for half in ([False, True] if device == 'cuda' else [False]) for b in [1, 8, 32, 128]]
speed = pd.DataFrame(rows, columns=['kiểu số', 'batch', 'ảnh/giây', 'bộ nhớ đỉnh (MB)'])
table(speed, 'Tốc độ và bộ nhớ khi tăng batch', fmt={'ảnh/giây': '{:.0f}', 'bộ nhớ đỉnh (MB)': '{:.0f}'}, best='ảnh/giây')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for kind, g in speed.groupby('kiểu số'):
    axes[0].plot(g['batch'], g['ảnh/giây'], marker='o', label=kind)
    axes[1].plot(g['batch'], g['bộ nhớ đỉnh (MB)'], marker='o', label=kind)
for ax, t in zip(axes, ['ảnh mỗi giây', 'bộ nhớ đỉnh (MB)']):
    ax.set_xscale('log', base=2); ax.set_xlabel('batch'); ax.set_title(t); ax.legend()
plt.tight_layout(); plt.show()
if device == 'cuda':
    v = speed.set_index(['kiểu số', 'batch'])
    note(f"Batch 128: float16 nhanh gấp <b>{v.loc[('float16', 128), 'ảnh/giây'] / v.loc[('float32', 128), 'ảnh/giây']:.1f}</b> lần float32 "
         f"và dùng {v.loc[('float16', 128), 'bộ nhớ đỉnh (MB)']:.0f} MB thay vì {v.loc[('float32', 128), 'bộ nhớ đỉnh (MB)']:.0f} MB. "
         f"Batch 1: float16 {v.loc[('float16', 1), 'ảnh/giây']:.0f} ảnh/giây, float32 {v.loc[('float32', 1), 'ảnh/giây']:.0f}; batch nhỏ thì GPU chưa bận, đổi kiểu số không giúp gì.")

## 7. Tóm tắt

In [ ]:
summary = [('accuracy chọn trong 10 lớp', f'{acc10:.3f}'),
           ('accuracy chọn trong 1000 lớp', f'{acc1000:.3f}'),
           ('lỗi tiền xử lý nặng nhất', f"{bad['cách']} → {bad['acc 1000 lớp']:.3f}")]
if device == 'cuda':
    fastest = speed.sort_values('ảnh/giây').groupby('kiểu số').tail(1).set_index('kiểu số')
    summary.append(('tốc độ cao nhất', ', '.join(f"{k}: {v:.0f} ảnh/giây" for k, v in fastest['ảnh/giây'].items())))
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** đổi `n=1000` ở mục 5 thành cả tập; thay `resnet50` bằng `efficientnet_b0` cùng `weights.transforms()` của nó; hoặc đổi ảnh ở mục 5 sang lớp khác để xem lỗi nào làm model đổi ý.